# Kaggle: parallel learning-curve experiments on T4 x2

This notebook runs a configurable experiment matrix on Kaggle's **two physical T4 GPUs**. `EXPERIMENTS_PER_GPU=4` means eight concurrent training processes (four sharing each GPU), not eight GPUs. If CUDA runs out of memory or throughput gets worse, reduce it to 2 or 1.

The output is a single live dashboard grouped by GPU. It shows only the experiments currently running and replaces itself every second, while detailed logs stay in files. Learning curves are logged separately to W&B at `LEARNING_CURVE_EVERY_STEPS`.

Before running: select **GPU T4 x2**, enable Internet, and add `WANDB_API_KEY` under **Add-ons > Secrets**.

In [ ]:
# ---- Experiment configuration: edit only this cell ----
K_UNITS = 64
TOTAL_EXPERIMENTS = 24       # first N experiments from the matrix (maximum 24)
EXPERIMENTS_PER_GPU = 4      # 4 x 2 physical GPUs = 8 concurrent processes
MAX_EPOCHS = 100
LEARNING_CURVE_EVERY_STEPS = 250  # W&B train/validation curves
DASHBOARD_EVERY_STEPS = 10        # progress only; does not add W&B points
GPUS = [0, 1]
WANDB_PROJECT = 'pc_dataset_eval'
OUTPUT_ROOT = '/kaggle/working/results'

REPOSITORY_URL = 'https://github.com/GianpieroTulipani/cirkit-circuits-training.git'
REPOSITORY_REF = None  # set a commit SHA for an exactly reproducible run

assert K_UNITS > 0
assert 1 <= TOTAL_EXPERIMENTS <= 24
assert EXPERIMENTS_PER_GPU > 0
assert MAX_EPOCHS > 0
assert LEARNING_CURVE_EVERY_STEPS > 0 and DASHBOARD_EVERY_STEPS > 0

In [ ]:
# Prepare the repository and dependencies. Kaggle Internet must be enabled.
from pathlib import Path
import os
import subprocess
import sys

workspace = Path('/kaggle/working')
repository = workspace / 'cirkit-circuits-training'
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repository)], check=True)
elif REPOSITORY_REF is None:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=repository, check=True)
if REPOSITORY_REF:
    subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=repository, check=True)
    subprocess.run(['git', 'checkout', REPOSITORY_REF], cwd=repository, check=True)
required_files = [
    repository / 'scripts' / 'run_learning_curves.py',
    repository / 'scripts' / 'evaluate_image_circuit.py',
    repository / 'notebooks' / 'data-driven-initialization.ipynb',
]
missing_files = [str(path.relative_to(repository)) for path in required_files if not path.exists()]
if missing_files:
    raise RuntimeError(
        'The selected GitHub revision does not contain the parallel launcher: '
        + ', '.join(missing_files)
        + '. Commit and push the local changes, then restart this notebook.'
    )
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repository, text=True).strip()
print(f'Using repository commit: {commit}')
os.chdir(repository)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

from kaggle_secrets import UserSecretsClient
import wandb
wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
if not wandb_key or not wandb_key.strip():
    raise RuntimeError('WANDB_API_KEY is missing or empty in Kaggle Secrets')
os.environ['WANDB_API_KEY'] = wandb_key.strip()
wandb.login(key=wandb_key.strip(), relogin=True)
del wandb_key
print('Repository, dependencies and W&B are ready.')

In [ ]:
# Hardware preflight: this notebook requires Kaggle GPU T4 x2.
import torch

gpu_count = torch.cuda.device_count()
print(f'Physical GPUs assigned by Kaggle: {gpu_count}')
for index in range(gpu_count):
    properties = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB')
if gpu_count != 2:
    raise RuntimeError('Select GPU T4 x2 in Kaggle before starting the experiments.')
if GPUS != [0, 1]:
    raise ValueError('For Kaggle T4 x2, keep GPUS = [0, 1].')

concurrent = gpu_count * EXPERIMENTS_PER_GPU
print(f'Concurrent processes: {concurrent} ({EXPERIMENTS_PER_GPU} per GPU)')
if EXPERIMENTS_PER_GPU > 2:
    print('WARNING: four k=64 models may not fit or may be slower. Reduce EXPERIMENTS_PER_GPU after an OOM.')

In [ ]:
# Compact preview: validate the campaign without listing every experiment.
command = [
    sys.executable, '-m', 'scripts.run_learning_curves',
    '--gpus', *[str(gpu) for gpu in GPUS],
    '--jobs-per-gpu', str(EXPERIMENTS_PER_GPU),
    '--max-experiments', str(TOTAL_EXPERIMENTS),
    '--k', str(K_UNITS),
    '--max-epochs', str(MAX_EPOCHS),
    '--validation-steps', str(LEARNING_CURVE_EVERY_STEPS),
    '--progress-steps', str(DASHBOARD_EVERY_STEPS),
    '--project', WANDB_PROJECT,
    '--output-dir', OUTPUT_ROOT,
]
subprocess.run([*command, '--dry-run'], cwd=repository, check=True)

## Start the campaign

The next cell shows one live dashboard, grouped by physical GPU, and replaces it every second. Only active experiments are visible. Stopping the cell also terminates the launcher and its child training processes. Detailed logs, checkpoints and `run-status.json` remain under `OUTPUT_ROOT`.

In [ ]:
# Run silently and render only the experiments currently occupying each GPU.
from datetime import datetime
from html import escape
from IPython.display import HTML, clear_output, display
import json
import re
import signal
import time

stamp = datetime.now().strftime('%Y%m%d-%H%M%S-%f')
campaign_dir = Path(OUTPUT_ROOT) / f'curves-{stamp}'
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
launcher_log = Path(OUTPUT_ROOT) / f'launcher-{stamp}.log'
status_file = campaign_dir / 'run-status.json'
campaign_command = [*command, '--run-dir', str(campaign_dir), '--no-progress']

def experiment_label(name):
    match = re.fullmatch(r'(mnist|fashion-mnist|cifar10)_(.+)_(data-init|random-init)_k(\d+)_seed(\d+)', name)
    if not match:
        return escape(name)
    dataset, structure, initialization, k_units, seed = match.groups()
    dataset = {'mnist': 'MNIST', 'fashion-mnist': 'Fashion-MNIST', 'cifar10': 'CIFAR-10'}[dataset]
    initialization = 'Data init' if initialization == 'data-init' else 'Random init'
    return f'<b>{dataset}</b> · {escape(structure)} · {initialization} · K={k_units} · seed={seed}'

def render_dashboard(status, running=True, exit_code=0):
    completed = status.get('completed', 0)
    failed = status.get('failed', 0)
    queued = status.get('queued', TOTAL_EXPERIMENTS)
    active = status.get('active', [])
    finished = completed + failed
    overall_percent = round(100 * finished / TOTAL_EXPERIMENTS)
    state = 'RUNNING' if running else ('LAUNCHER ERROR' if exit_code else ('COMPLETED' if failed == 0 else 'FINISHED WITH ERRORS'))
    color = '#2563eb' if running else ('#16a34a' if not exit_code and failed == 0 else '#dc2626')
    parts = [
        '<div style="font-family:system-ui,sans-serif;max-width:1200px">',
        f'<div style="display:flex;justify-content:space-between;align-items:center"><h3 style="margin:0">Learning curves</h3><b style="color:{color}">{state}</b></div>',
        f'<div style="margin:8px 0 4px"><b>{finished}/{TOTAL_EXPERIMENTS}</b> finished &nbsp;·&nbsp; <b>{len(active)}</b> running &nbsp;·&nbsp; <b>{queued}</b> queued &nbsp;·&nbsp; <b>{failed}</b> failed</div>',
        f'<div style="height:7px;background:#e5e7eb;border-radius:4px;margin-bottom:14px"><div style="height:7px;width:{overall_percent}%;background:{color};border-radius:4px"></div></div>',
        '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:12px">',
    ]
    for gpu in GPUS:
        gpu_runs = sorted(
            (run for run in active if str(run['gpu']) == str(gpu)),
            key=lambda run: run['slot'],
        )
        parts.append(f'<section style="border:1px solid #d1d5db;border-radius:10px;padding:12px"><div style="display:flex;justify-content:space-between"><b>GPU {gpu}</b><span>{len(gpu_runs)}/{EXPERIMENTS_PER_GPU} active</span></div>')
        if not gpu_runs:
            parts.append('<div style="color:#6b7280;padding:18px 0 6px;text-align:center">No active experiment</div>')
        for local_slot, run in enumerate(gpu_runs, start=1):
            step = int(run.get('step') or 0)
            maximum = int(run.get('max_steps') or 0)
            percent = min(100, round(100 * step / maximum)) if maximum else 0
            elapsed = float(run.get('elapsed_seconds') or 0) / 60
            progress = f'{step}/{maximum} steps · {percent}%' if maximum else 'Initializing'
            parts.append(
                f'<div style="border-top:1px solid #e5e7eb;margin-top:10px;padding-top:10px">'
                f'<div style="display:flex;gap:8px;align-items:center"><span style="background:#eef2ff;color:#3730a3;border-radius:5px;padding:2px 6px;font-size:12px">Slot {local_slot}</span><span>{experiment_label(run["name"])}</span></div>'
                f'<div style="display:flex;justify-content:space-between;margin:7px 0 3px;font-size:13px"><span>{progress}</span><span>{elapsed:.1f} min</span></div>'
                f'<div style="height:8px;background:#e5e7eb;border-radius:4px"><div style="height:8px;width:{percent}%;background:#2563eb;border-radius:4px"></div></div></div>'
            )
        parts.append('</section>')
    parts.append('</div></div>')
    clear_output(wait=True)
    display(HTML(''.join(parts)))

status = {'queued': TOTAL_EXPERIMENTS, 'completed': 0, 'failed': 0, 'active': []}
with launcher_log.open('w', encoding='utf-8') as log_stream:
    launcher = subprocess.Popen(
        campaign_command, cwd=repository, stdout=log_stream, stderr=subprocess.STDOUT, text=True
    )
    try:
        while launcher.poll() is None:
            if status_file.exists():
                try:
                    status = json.loads(status_file.read_text(encoding='utf-8'))
                except json.JSONDecodeError:
                    pass
            render_dashboard(status, running=True)
            time.sleep(1)
    except KeyboardInterrupt:
        launcher.send_signal(signal.SIGINT)
        try:
            launcher.wait(timeout=20)
        except subprocess.TimeoutExpired:
            launcher.kill()
            launcher.wait()
        raise

if status_file.exists():
    status = json.loads(status_file.read_text(encoding='utf-8'))
render_dashboard(status, running=False, exit_code=launcher.returncode)
if launcher.returncode:
    print(f'Launcher exited with code {launcher.returncode}. Details: {launcher_log}')
else:
    print(f'Artifacts: {campaign_dir}')

In [ ]:
# Compact final summary; full details remain in run-status.json and per-run logs.
import json

if 'campaign_dir' not in globals():
    run_directories = sorted(Path(OUTPUT_ROOT).glob('curves-*'))
    if not run_directories:
        raise FileNotFoundError(f'No run directory found under {OUTPUT_ROOT}')
    campaign_dir = run_directories[-1]
latest_run = campaign_dir
status_file = latest_run / 'run-status.json'
status = json.loads(status_file.read_text(encoding='utf-8'))
print(f'Latest run: {latest_run}')
print(f"queued={status['queued']} completed={status['completed']} failed={status['failed']}")
for result in status['results']:
    if result['status'] == 'failed':
        print(f"FAILED GPU {result['gpu']}: {result['name']} — {result['log']}")